In [ ]:
import sys

print(sys.executable)

In [ ]:
import pandas as pd

calendar = pd.read_csv("../data/raw/calendar.csv")
calendar.shape
calendar.head()

In [ ]:
calendar.info()

In [ ]:
sell_prices = pd.read_csv("../data/raw/sell_prices.csv")
sell_prices.shape

In [ ]:
sell_prices.head()

In [ ]:
sell_prices.info()

In [ ]:
print("Stores:", sell_prices["store_id"].nunique())
print("Items:", sell_prices["item_id"].nunique())
print("Weeks:", sell_prices["wm_yr_wk"].nunique())

print("\nMissing values:")
print(sell_prices.isnull().sum())

print("\nDuplicate rows:", sell_prices.duplicated().sum())

In [ ]:
sell_prices["store_id"] = sell_prices["store_id"].astype("category")
sell_prices["item_id"] = sell_prices["item_id"].astype("category")

sell_prices["wm_yr_wk"] = pd.to_numeric(
    sell_prices["wm_yr_wk"],
    downcast="integer"
)

sell_prices["sell_price"] = pd.to_numeric(
    sell_prices["sell_price"],
    downcast="float"
)

In [ ]:
sell_prices.info()

In [ ]:
sell_prices["sell_price"].describe()

In [ ]:
sales = pd.read_csv("../data/raw/sales_train_evaluation.csv")

sales.shape

In [ ]:
sales.head()

In [ ]:
sales.info(memory_usage="deep")

In [ ]:
day_cols = [col for col in sales.columns if col.startswith("d_")]

sales[day_cols].max().max()

In [ ]:
sales[day_cols] = sales[day_cols].astype("int16")

In [ ]:
id_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

for col in id_cols:
    sales[col] = sales[col].astype("category")

In [ ]:
sales.info(memory_usage="deep")

In [ ]:
print("Categories:", sales["cat_id"].nunique())
print("Departments:", sales["dept_id"].nunique())
print("States:", sales["state_id"].nunique())
print("Stores:", sales["store_id"].nunique())

print("\nCategories:")
print(sales["cat_id"].value_counts())

print("\nDepartments:")
print(sales["dept_id"].value_counts())

print("\nStores:")
print(sales["store_id"].value_counts())

In [ ]:
sales["total_sales"] = sales[day_cols].sum(axis=1)

In [ ]:
category_sales = (
    sales.groupby("cat_id", observed=True)["total_sales"]
    .sum()
    .sort_values(ascending=False)
)

category_sales

In [ ]:
store_sales = (
    sales.groupby("store_id", observed=True)["total_sales"]
    .sum()
    .sort_values(ascending=False)
)

store_sales

In [ ]:
department_sales = (
    sales.groupby("dept_id", observed=True)["total_sales"]
    .sum()
    .sort_values(ascending=False)
)

department_sales

In [ ]:
import matplotlib.pyplot as plt

category_sales.sort_values().plot(
    kind="barh",
    figsize=(8, 4)
)

plt.title("Total Unit Sales by Category")
plt.xlabel("Total Units Sold")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

In [ ]:
store_sales.sort_values().plot(
    kind="barh",
    figsize=(8, 6)
)

plt.title("Total Unit Sales by Store")
plt.xlabel("Total Units Sold")
plt.ylabel("Store")
plt.tight_layout()
plt.show()

In [ ]:
department_sales.sort_values().plot(
    kind="barh",
    figsize=(8, 5)
)

plt.title("Total Unit Sales by Department")
plt.xlabel("Total Units Sold")
plt.ylabel("Department")
plt.tight_layout()
plt.show()

In [ ]:
calendar["date"] = pd.to_datetime(calendar["date"])

calendar[["date", "d", "wm_yr_wk"]].head()

In [ ]:
day_cols[:5]

In [ ]:
daily_sales = sales[day_cols].sum(axis=0).reset_index()

daily_sales.columns = ["d", "total_sales"]

daily_sales.head()

In [ ]:
daily_sales = daily_sales.merge(
    calendar[["d", "date", "weekday", "wday", "month", "year"]],
    on="d",
    how="left"
)

daily_sales.head()

In [ ]:
daily_sales.info()

In [ ]:
daily_sales[["date", "total_sales"]].head()

In [ ]:
plt.figure(figsize=(15, 6))

plt.plot(
    daily_sales["date"],
    daily_sales["total_sales"]
)

plt.title("Daily Total Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Total Units Sold")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
daily_sales.nsmallest(10, "total_sales")[
    ["date", "weekday", "total_sales"]
]

In [ ]:
calendar[
    calendar["date"].dt.strftime("%m-%d") == "12-25"
][
    ["date", "weekday", "event_name_1", "event_type_1"]
]

In [ ]:
daily_sales["rolling_7"] = (
    daily_sales["total_sales"]
    .rolling(window=7)
    .mean()
)

daily_sales["rolling_28"] = (
    daily_sales["total_sales"]
    .rolling(window=28)
    .mean()
)

In [ ]:
plt.figure(figsize=(15, 6))

plt.plot(
    daily_sales["date"],
    daily_sales["total_sales"],
    alpha=0.25,
    label="Daily Sales"
)

plt.plot(
    daily_sales["date"],
    daily_sales["rolling_7"],
    label="7-Day Moving Average"
)

plt.plot(
    daily_sales["date"],
    daily_sales["rolling_28"],
    linewidth=2,
    label="28-Day Moving Average"
)

plt.title("Daily Sales with 7-Day and 28-Day Moving Averages")
plt.xlabel("Date")
plt.ylabel("Total Units Sold")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
weekday_sales = (
    daily_sales.groupby("weekday")["total_sales"]
    .agg(["mean", "median"])
)

weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

weekday_sales = weekday_sales.reindex(weekday_order)

weekday_sales.round(2)

In [ ]:
weekday_sales["mean"].plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Average Daily Sales by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average Units Sold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
daily_sales = daily_sales.merge(
    calendar[
        ["d", "event_name_1", "event_type_1",
         "event_name_2", "event_type_2",
         "snap_CA", "snap_TX", "snap_WI"]
    ],
    on="d",
    how="left"
)

In [ ]:
daily_sales = daily_sales.merge(
    calendar[
        ["d", "event_name_1", "event_type_1",
         "event_name_2", "event_type_2",
         "snap_CA", "snap_TX", "snap_WI"]
    ],
    on="d",
    how="left"
)
daily_sales.head()

In [ ]:
daily_sales["event_type"] = (
    daily_sales["event_type_1"]
    .fillna("No Event")
)

daily_sales["event_type"].value_counts()

In [ ]:
daily_sales["event_type_2"].value_counts(dropna=False)

In [ ]:
daily_sales["event_type_2_y"].value_counts(dropna=False)

In [ ]:
[col for col in daily_sales.columns if "event" in col]

In [ ]:
event_cols = [col for col in daily_sales.columns if "event_" in col]

daily_sales = daily_sales.drop(columns=event_cols)

In [ ]:
[col for col in daily_sales.columns if "event" in col]

In [ ]:
daily_sales = daily_sales.merge(
    calendar[
        [
            "d",
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2"
        ]
    ],
    on="d",
    how="left"
)

In [ ]:
[col for col in daily_sales.columns if "event" in col]

In [ ]:
daily_sales["event_type"] = (
    daily_sales["event_type_1"]
    .fillna(daily_sales["event_type_2"])
    .fillna("No Event")
)

In [ ]:
daily_sales["event_type"].value_counts()

In [ ]:
event_sales_summary = (
    daily_sales
    .groupby("event_type")["total_sales"]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
)

event_sales_summary.round(2)

In [ ]:
no_event_mean = event_sales_summary.loc["No Event", "mean"]

event_sales_summary["difference_vs_no_event_%"] = (
    (event_sales_summary["mean"] - no_event_mean)
    / no_event_mean * 100
)

event_sales_summary.round(2)

In [ ]:
event_sales_summary = (
    daily_sales
    .groupby("event_type")["total_sales"]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
)

event_sales_summary.round(2)

In [ ]:
no_event_mean = event_sales_summary.loc["No Event", "mean"]

event_sales_summary["difference_vs_no_event_%"] = (
    (event_sales_summary["mean"] - no_event_mean)
    / no_event_mean * 100
)

event_sales_summary.round(2)

In [ ]:
state_daily_sales = {}

for state in ["CA", "TX", "WI"]:
    state_mask = sales["state_id"] == state

    state_daily_sales[state] = (
        sales.loc[state_mask, day_cols]
        .sum(axis=0)
    )

state_daily_sales["CA"].head()

In [ ]:
state_daily_sales = {}

for state in ["CA", "TX", "WI"]:
    state_mask = sales["state_id"] == state

    state_daily_sales[state] = (
        sales.loc[state_mask, day_cols]
        .sum(axis=0)
    )

state_daily_sales["TX"].head()

In [ ]:
state_sales_df = pd.DataFrame({
    "d": day_cols,
    "CA_sales": state_daily_sales["CA"].values,
    "TX_sales": state_daily_sales["TX"].values,
    "WI_sales": state_daily_sales["WI"].values
})

state_sales_df.head()

In [ ]:
state_sales_df = state_sales_df.merge(
    calendar[
        ["d", "date", "snap_CA", "snap_TX", "snap_WI"]
    ],
    on="d",
    how="left"
)

state_sales_df.head()

In [ ]:
snap_results = []

for state in ["CA", "TX", "WI"]:
    result = (
        state_sales_df
        .groupby(f"snap_{state}")[f"{state}_sales"]
        .agg(["count", "mean", "median"])
    )

    no_snap_mean = result.loc[0, "mean"]
    snap_mean = result.loc[1, "mean"]

    difference_pct = (
        (snap_mean - no_snap_mean)
        / no_snap_mean * 100
    )

    snap_results.append({
        "state": state,
        "no_snap_mean": no_snap_mean,
        "snap_mean": snap_mean,
        "difference_pct": difference_pct
    })

snap_summary = pd.DataFrame(snap_results)

snap_summary.round(2)

In [ ]:
price_variation = (
    sell_prices
    .groupby(["store_id", "item_id"], observed=True)["sell_price"]
    .nunique()
)

price_variation.describe()

In [ ]:
changed_price_pct = (
    (price_variation > 1).mean() * 100
)

print(f"Price changed: {changed_price_pct:.2f}%")

In [ ]:
price_variation.value_counts().sort_index().head(15)

In [ ]:
sell_prices = sell_prices.sort_values(
    ["store_id", "item_id", "wm_yr_wk"]
).copy()

In [ ]:
sell_prices["previous_price"] = (
    sell_prices
    .groupby(["store_id", "item_id"], observed=True)["sell_price"]
    .shift(1)
)

In [ ]:
sell_prices["price_change"] = (
    sell_prices["sell_price"]
    - sell_prices["previous_price"]
)

In [ ]:
sell_prices["price_change_pct"] = (
    sell_prices["price_change"]
    / sell_prices["previous_price"] * 100
)

In [ ]:
price_change_counts = pd.Series({
    "Price Decreased": (sell_prices["price_change"] < 0).sum(),
    "No Change": (sell_prices["price_change"] == 0).sum(),
    "Price Increased": (sell_prices["price_change"] > 0).sum()
})

price_change_counts

In [ ]:
sell_prices["price_change_pct"].describe()

In [ ]:
sell_prices.nlargest(10, "price_change_pct")[
    [
        "store_id",
        "item_id",
        "wm_yr_wk",
        "previous_price",
        "sell_price",
        "price_change",
        "price_change_pct"
    ]
]

In [ ]:
sell_prices.nsmallest(10, "price_change_pct")[
    [
        "store_id",
        "item_id",
        "wm_yr_wk",
        "previous_price",
        "sell_price",
        "price_change",
        "price_change_pct"
    ]
]

In [ ]:
sell_prices["price_direction"] = "No Change"

sell_prices.loc[
    sell_prices["price_change"] > 0,
    "price_direction"
] = "Increase"

sell_prices.loc[
    sell_prices["price_change"] < 0,
    "price_direction"
] = "Decrease"

In [ ]:
sell_prices["price_direction"].value_counts()

In [ ]:
extreme_low_prices = sell_prices[
    sell_prices["sell_price"] <= 0.10
]

print("Records with price <= $0.10:", len(extreme_low_prices))
print(
    "Unique item-store combinations:",
    extreme_low_prices[["store_id", "item_id"]]
    .drop_duplicates()
    .shape[0]
)

In [ ]:
print(
    f"Share of price records <= $0.10: "
    f"{len(extreme_low_prices) / len(sell_prices) * 100:.4f}%"
)

In [ ]:
day_week_map = calendar[
    calendar["d"].isin(day_cols)
][["d", "wm_yr_wk"]]

day_week_map.head(10)

In [ ]:
day_week_map.groupby("wm_yr_wk").size().value_counts().sort_index()

In [ ]:
weekly_sales_list = []

for week in weeks:
    week_days = day_week_map.loc[
        day_week_map["wm_yr_wk"] == week,
        "d"
    ].tolist()

    week_sales = sales[week_days].sum(axis=1)

    temp = pd.DataFrame({
        "store_id": sales["store_id"],
        "item_id": sales["item_id"],
        "wm_yr_wk": week,
        "weekly_sales": week_sales
    })

    weekly_sales_list.append(temp)

In [ ]:
weeks = day_week_map["wm_yr_wk"].unique()

weekly_sales_list = []

for week in weeks:
    week_days = day_week_map.loc[
        day_week_map["wm_yr_wk"] == week,
        "d"
    ].tolist()

    week_sales = sales[week_days].sum(axis=1)

    temp = pd.DataFrame({
        "store_id": sales["store_id"],
        "item_id": sales["item_id"],
        "wm_yr_wk": week,
        "weekly_sales": week_sales
    })

    weekly_sales_list.append(temp)

In [ ]:
len (weeks)

In [ ]:
weekly_sales = pd.concat(
    weekly_sales_list,
    ignore_index=True
)

weekly_sales.shape

In [ ]:
weekly_sales.head()

In [ ]:
weekly_sales.info(memory_usage="deep")

In [ ]:
weekly_sales["weekly_sales"].max()

In [ ]:
weekly_sales["wm_yr_wk"] = weekly_sales["wm_yr_wk"].astype("int16")
weekly_sales["weekly_sales"] = weekly_sales["weekly_sales"].astype("int16")

In [ ]:
weekly_sales.info(memory_usage="deep")

In [ ]:
print(weekly_sales[["store_id", "item_id", "wm_yr_wk"]].dtypes)
print()
print(sell_prices[["store_id", "item_id", "wm_yr_wk"]].dtypes)

In [ ]:
weekly_price_sales = weekly_sales.merge(
    sell_prices[
        [
            "store_id",
            "item_id",
            "wm_yr_wk",
            "sell_price",
            "previous_price",
            "price_change",
            "price_change_pct",
            "price_direction"
        ]
    ],
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

In [ ]:
print("Shape:", weekly_price_sales.shape)

print(
    "Missing price:",
    weekly_price_sales["sell_price"].isna().sum()
)

weekly_price_sales.head()

In [ ]:
missing_price = weekly_price_sales[
    weekly_price_sales["sell_price"].isna()
]

missing_price["weekly_sales"].describe()

In [ ]:
print(
    "Missing price & zero sales:",
    ((weekly_price_sales["sell_price"].isna()) &
     (weekly_price_sales["weekly_sales"] == 0)).sum()
)

print(
    "Missing price & positive sales:",
    ((weekly_price_sales["sell_price"].isna()) &
     (weekly_price_sales["weekly_sales"] > 0)).sum()
)

In [ ]:
price_sales_active = weekly_price_sales[
    weekly_price_sales["sell_price"].notna()
].copy()

price_sales_active.shape

In [ ]:
price_sales_active = price_sales_active.sort_values(
    ["store_id", "item_id", "wm_yr_wk"]
)

price_sales_active["previous_week_sales"] = (
    price_sales_active
    .groupby(["store_id", "item_id"], observed=True)["weekly_sales"]
    .shift(1)
)

price_sales_active["sales_change"] = (
    price_sales_active["weekly_sales"]
    - price_sales_active["previous_week_sales"]
)

In [ ]:
price_demand_summary = (
    price_sales_active
    .groupby("price_direction", observed=True)
    .agg(
        weeks=("weekly_sales", "count"),
        avg_sales=("weekly_sales", "mean"),
        avg_previous_sales=("previous_week_sales", "mean"),
        avg_sales_change=("sales_change", "mean"),
        median_sales_change=("sales_change", "median")
    )
)

price_demand_summary.round(2)

In [ ]:
week_order = (
    calendar[["wm_yr_wk"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

week_order["week_index"] = range(len(week_order))

week_order.head()

In [ ]:
price_sales_active = price_sales_active.merge(
    week_order,
    on="wm_yr_wk",
    how="left"
)

price_sales_active = price_sales_active.sort_values(
    ["store_id", "item_id", "week_index"]
)

price_sales_active["previous_week_index"] = (
    price_sales_active
    .groupby(["store_id", "item_id"], observed=True)["week_index"]
    .shift(1)
)

price_sales_active["week_gap"] = (
    price_sales_active["week_index"]
    - price_sales_active["previous_week_index"]
)

In [ ]:
price_sales_active["week_gap"].value_counts().head(10)

In [ ]:
# Her ürün-mağaza kombinasyonu için normal/ortalama fiyat
weekly_sales["avg_price_item_store"] = (
    weekly_sales
    .groupby(["item_id", "store_id"])["sell_price"]
    .transform("median")
)

# O haftaki fiyatın normal fiyata göre değişimi
weekly_sales["price_ratio"] = (
    weekly_sales["sell_price"] /
    weekly_sales["avg_price_item_store"]
)

weekly_sales[
    ["item_id", "store_id", "sell_price",
     "avg_price_item_store", "price_ratio", "weekly_sales"]
].head()

In [ ]:
weekly_sales.columns.tolist()

In [ ]:
weekly_sales.head()

In [ ]:
get_ipython().run_line_magic('whos', 'DataFrame')

In [ ]:
price_sales_active.columns.tolist()

In [ ]:
price_sales_active.head()

In [ ]:
price_demand_summary

In [ ]:
price_sales_active.loc[
    price_sales_active["price_direction"] != "No Change",
    "price_change_pct"
].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99])

In [ ]:
price_change_analysis = price_sales_active[
    (price_sales_active["price_direction"] != "No Change") &
    (price_sales_active["price_change_pct"].between(-50, 100)) &
    (price_sales_active["week_gap"] == 1)
].copy()

price_change_analysis.shape

In [ ]:
price_change_analysis["price_change_pct"].describe()

In [ ]:
price_change_analysis.shape

In [ ]:
price_sales_active = price_sales_active.sort_values(
    ["store_id", "item_id", "week_index"]
).reset_index(drop=True)

In [ ]:
group = price_sales_active.groupby(
    ["store_id", "item_id"]
)["weekly_sales"]

price_sales_active["lag_1"] = group.shift(1)
price_sales_active["lag_2"] = group.shift(2)
price_sales_active["lag_4"] = group.shift(4)
price_sales_active["lag_8"] = group.shift(8)

In [ ]:
price_sales_active[
    [
        "store_id",
        "item_id",
        "wm_yr_wk",
        "week_index",
        "weekly_sales",
        "lag_1",
        "lag_2",
        "lag_4",
        "lag_8"
    ]
].head(15)

In [ ]:
group_cols = ["store_id", "item_id"]

price_sales_active["rolling_mean_4"] = (
    price_sales_active
    .groupby(group_cols)["weekly_sales"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

price_sales_active["rolling_mean_8"] = (
    price_sales_active
    .groupby(group_cols)["weekly_sales"]
    .transform(lambda x: x.shift(1).rolling(8).mean())
)

In [ ]:
price_sales_active[
    [
        "store_id",
        "item_id",
        "wm_yr_wk",
        "weekly_sales",
        "lag_1",
        "rolling_mean_4",
        "rolling_mean_8",
        "rolling_std_4"
    ]
].head(12)

In [ ]:
price_sales_active[
    [
        "store_id",
        "item_id",
        "wm_yr_wk",
        "weekly_sales",
        "lag_1",
        "rolling_mean_4",
        "rolling_mean_8",
    ]
].head(12)

In [ ]:
price_sales_active["rolling_std_4"] = (
    price_sales_active
    .groupby(["store_id", "item_id"])["weekly_sales"]
    .transform(lambda x: x.shift(1).rolling(4).std())
)

In [ ]:
price_sales_active[
    ["weekly_sales", "rolling_mean_4", "rolling_mean_8", "rolling_std_4"]
].head(10)

In [ ]:
calendar.columns.tolist()

In [ ]:
calendar["has_event"] = (
    calendar["event_name_1"].notna() |
    calendar["event_name_2"].notna()
).astype(int)

In [ ]:
weekly_calendar = (
    calendar
    .groupby("wm_yr_wk")
    .agg(
        month=("month", "first"),
        year=("year", "first"),
        has_event=("has_event", "max"),
        snap_CA=("snap_CA", "max"),
        snap_TX=("snap_TX", "max"),
        snap_WI=("snap_WI", "max")
    )
    .reset_index()
)

weekly_calendar.head()

In [ ]:
weekly_calendar = (
    calendar
    .groupby("wm_yr_wk")
    .agg(
        month=("month", "first"),
        year=("year", "first"),
        event_days=("has_event", "sum"),
        snap_CA_days=("snap_CA", "sum"),
        snap_TX_days=("snap_TX", "sum"),
        snap_WI_days=("snap_WI", "sum")
    )
    .reset_index()
)

weekly_calendar.head()

In [ ]:
price_sales_active = price_sales_active.merge(
    weekly_calendar,
    on="wm_yr_wk",
    how="left"
)

In [ ]:
import numpy as np

price_sales_active["snap_days"] = np.select(
    [
        price_sales_active["store_id"].str.startswith("CA"),
        price_sales_active["store_id"].str.startswith("TX"),
        price_sales_active["store_id"].str.startswith("WI")
    ],
    [
        price_sales_active["snap_CA_days"],
        price_sales_active["snap_TX_days"],
        price_sales_active["snap_WI_days"]
    ],
    default=0
)

In [ ]:
price_sales_active[
    [
        "store_id",
        "wm_yr_wk",
        "weekly_sales",
        "month",
        "year",
        "event_days",
        "snap_days"
    ]
].head(10)

In [ ]:
event_types = ["Religious", "National", "Cultural", "Sporting"]

for event_type in event_types:
    calendar[f"event_{event_type.lower()}"] = (
        (calendar["event_type_1"] == event_type) |
        (calendar["event_type_2"] == event_type)
    ).astype(int)

In [ ]:
weekly_event_types = (
    calendar
    .groupby("wm_yr_wk")
    .agg(
        religious_days=("event_religious", "sum"),
        national_days=("event_national", "sum"),
        cultural_days=("event_cultural", "sum"),
        sporting_days=("event_sporting", "sum")
    )
    .reset_index()
)

weekly_event_types.head()

In [ ]:
price_sales_active = price_sales_active.merge(
    weekly_event_types,
    on="wm_yr_wk",
    how="left"
)

In [ ]:
price_sales_active[
    [
        "event_days",
        "religious_days",
        "national_days",
        "cultural_days",
        "sporting_days"
    ]
].isna().sum()

In [ ]:
features = [
    "store_id",
    "item_id",

    # Demand history
    "lag_1",
    "lag_2",
    "lag_4",
    "lag_8",
    "rolling_mean_4",
    "rolling_mean_8",
    "rolling_std_4",

    # Price
    "sell_price",
    "price_change_pct",

    # Calendar
    "month",
    "year",
    "event_days",
    "religious_days",
    "national_days",
    "cultural_days",
    "sporting_days",
    "snap_days"
]

target = "weekly_sales"

In [ ]:
price_sales_active[features + [target]].isna().sum().sort_values(ascending=False)

In [ ]:
model_df = price_sales_active[
    ["wm_yr_wk", "week_index"] + features + [target]
].dropna().copy()

model_df.shape

In [ ]:
model_df[
    [
        "store_id",
        "item_id",
        "wm_yr_wk",
        "week_index",
        "weekly_sales",
        "lag_1",
        "lag_8",
        "rolling_mean_4",
        "rolling_mean_8",
        "sell_price",
        "snap_days"
    ]
].head()

In [ ]:
model_df["week_index"].agg(["min", "max", "nunique"])

In [ ]:
max_week = model_df["week_index"].max()

test_start = max_week - 3
val_start = test_start - 4

train_df = model_df[
    model_df["week_index"] < val_start
].copy()

val_df = model_df[
    (model_df["week_index"] >= val_start) &
    (model_df["week_index"] < test_start)
].copy()

test_df = model_df[
    model_df["week_index"] >= test_start
].copy()

In [ ]:
print(
    "Train:",
    train_df.shape,
    train_df["week_index"].min(),
    "-",
    train_df["week_index"].max()
)

print(
    "Validation:",
    val_df.shape,
    val_df["week_index"].min(),
    "-",
    val_df["week_index"].max()
)

print(
    "Test:",
    test_df.shape,
    test_df["week_index"].min(),
    "-",
    test_df["week_index"].max()
)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_val = val_df["weekly_sales"]
y_pred_naive = val_df["lag_1"]

mae = mean_absolute_error(y_val, y_pred_naive)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_naive))

print(f"Naive Baseline MAE: {mae:.4f}")
print(f"Naive Baseline RMSE: {rmse:.4f}")

In [ ]:
import sklearn
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_val = val_df["weekly_sales"]
y_pred_naive = val_df["lag_1"]

mae = mean_absolute_error(y_val, y_pred_naive)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_naive))

print(f"Naive Baseline MAE: {mae:.4f}")
print(f"Naive Baseline RMSE: {rmse:.4f}")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_val = val_df["weekly_sales"]
y_pred_naive = val_df["lag_1"]

mae = mean_absolute_error(y_val, y_pred_naive)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_naive))

print(f"Naive Baseline MAE: {mae:.4f}")
print(f"Naive Baseline RMSE: {rmse:.4f}")

In [ ]:
pip install scikit-learn

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_val = val_df["weekly_sales"]
y_pred_naive = val_df["lag_1"]

mae = mean_absolute_error(y_val, y_pred_naive)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_naive))

print(f"Naive Baseline MAE: {mae:.4f}")
print(f"Naive Baseline RMSE: {rmse:.4f}")

In [ ]:
get_ipython().system(' pip install scikit-learn')

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_val = val_df["weekly_sales"]
y_pred_naive = val_df["lag_1"]

mae = mean_absolute_error(y_val, y_pred_naive)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_naive))

print(f"Naive Baseline MAE: {mae:.4f}")
print(f"Naive Baseline RMSE: {rmse:.4f}")

In [ ]:
model_features = [
    "store_id",
    "item_id",

    "lag_1",
    "lag_2",
    "lag_4",
    "lag_8",
    "rolling_mean_4",
    "rolling_mean_8",
    "rolling_std_4",

    "sell_price",
    "price_change_pct",

    "month",
    "year",
    "event_days",
    "religious_days",
    "national_days",
    "cultural_days",
    "sporting_days",
    "snap_days"
]

X_train = train_df[model_features].copy()
y_train = train_df["weekly_sales"]

X_val = val_df[model_features].copy()
y_val = val_df["weekly_sales"]

In [ ]:
categorical_features = ["store_id", "item_id"]

for col in categorical_features:
    X_train[col] = X_train[col].astype("category")
    X_val[col] = X_val[col].astype("category")

In [ ]:
print(X_train.shape)
print(X_val.shape)

print("\nDtypes:")
print(X_train.dtypes)

In [ ]:
import lightgbm as lgb
from lightgbm import LGBMRegressor

lgb_model = LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    categorical_feature=["store_id", "item_id"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

In [ ]:
get_ipython().system(' pip install lightgbm')
import lightgbm as lgb
from lightgbm import LGBMRegressor

lgb_model = LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    categorical_feature=["store_id", "item_id"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

In [ ]:
y_pred_lgb = lgb_model.predict(
    X_val,
    num_iteration=lgb_model.best_iteration_
)

lgb_mae = mean_absolute_error(y_val, y_pred_lgb)
lgb_rmse = np.sqrt(mean_squared_error(y_val, y_pred_lgb))

print(f"LightGBM MAE:  {lgb_mae:.4f}")
print(f"LightGBM RMSE: {lgb_rmse:.4f}")
print(f"Best iteration: {lgb_model.best_iteration_}")

In [ ]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": lgb_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance

In [ ]:
lgb_model_v2 = LGBMRegressor(
    objective="regression",
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1
)

lgb_model_v2.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    categorical_feature=["store_id", "item_id"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=75),
        lgb.log_evaluation(period=100)
    ]
)

In [ ]:
y_pred_v2 = lgb_model_v2.predict(
    X_val,
    num_iteration=lgb_model_v2.best_iteration_
)

v2_mae = mean_absolute_error(y_val, y_pred_v2)
v2_rmse = np.sqrt(mean_squared_error(y_val, y_pred_v2))

print(f"Naive        MAE: 3.9089 | RMSE: 8.4082")
print(f"LightGBM v1  MAE: 3.2782 | RMSE: 6.9728")
print(f"LightGBM v2  MAE: {v2_mae:.4f} | RMSE: {v2_rmse:.4f}")
print(f"Best iteration: {lgb_model_v2.best_iteration_}")

In [ ]:
price_sales_active = price_sales_active.sort_values(
    ["store_id", "item_id", "week_index"]
).reset_index(drop=True)

price_sales_active["lag_52"] = (
    price_sales_active
    .groupby(["store_id", "item_id"])["weekly_sales"]
    .shift(52)
)

In [ ]:
price_sales_active[
    ["store_id", "item_id", "week_index", "weekly_sales", "lag_52"]
].head(57)

In [ ]:
features_v3 = features + ["lag_52"]

model_df_v3 = price_sales_active[
    ["wm_yr_wk", "week_index"] + features_v3 + [target]
].dropna().copy()

In [ ]:
train_v3 = model_df_v3[model_df_v3["week_index"] < 270].copy()

val_v3 = model_df_v3[
    model_df_v3["week_index"].between(270, 273)
].copy()

test_v3 = model_df_v3[
    model_df_v3["week_index"].between(274, 277)
].copy()

print("Train:", train_v3.shape,
      train_v3["week_index"].min(), "-", train_v3["week_index"].max())

print("Validation:", val_v3.shape,
      val_v3["week_index"].min(), "-", val_v3["week_index"].max())

print("Test:", test_v3.shape,
      test_v3["week_index"].min(), "-", test_v3["week_index"].max())

In [ ]:
lag52_lookup = price_sales_active[
    ["store_id", "item_id", "week_index", "weekly_sales"]
].copy()

lag52_lookup["week_index"] = lag52_lookup["week_index"] + 52

lag52_lookup = lag52_lookup.rename(
    columns={"weekly_sales": "lag_52_true"}
)

price_sales_active = price_sales_active.merge(
    lag52_lookup,
    on=["store_id", "item_id", "week_index"],
    how="left"
)

In [ ]:
(
    price_sales_active["lag_52"].notna().sum(),
    price_sales_active["lag_52_true"].notna().sum()
)

In [ ]:
mask = (
    price_sales_active["lag_52"].notna() &
    price_sales_active["lag_52_true"].notna()
)

(
    price_sales_active.loc[mask, "lag_52"]
    != price_sales_active.loc[mask, "lag_52_true"]
).sum()

In [ ]:
price_sales_active.drop(columns="lag_52_true", inplace=True)

In [ ]:
model_features_v3 = model_features + ["lag_52"]

X_train_v3 = train_v3[model_features_v3].copy()
y_train_v3 = train_v3["weekly_sales"]

X_val_v3 = val_v3[model_features_v3].copy()
y_val_v3 = val_v3["weekly_sales"]

for col in ["store_id", "item_id"]:
    X_train_v3[col] = X_train_v3[col].astype("category")
    X_val_v3[col] = X_val_v3[col].astype("category")

In [ ]:
lgb_model_v3 = LGBMRegressor(
    objective="regression",
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1
)

lgb_model_v3.fit(
    X_train_v3,
    y_train_v3,
    eval_set=[(X_val_v3, y_val_v3)],
    eval_metric="rmse",
    categorical_feature=["store_id", "item_id"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=75),
        lgb.log_evaluation(period=100)
    ]
)

In [ ]:
y_pred_v3 = lgb_model_v3.predict(
    X_val_v3,
    num_iteration=lgb_model_v3.best_iteration_
)

v3_mae = mean_absolute_error(y_val_v3, y_pred_v3)
v3_rmse = np.sqrt(mean_squared_error(y_val_v3, y_pred_v3))

print(f"LightGBM v3 MAE:  {v3_mae:.4f}")
print(f"LightGBM v3 RMSE: {v3_rmse:.4f}")
print(f"Best iteration:   {lgb_model_v3.best_iteration_}")

In [ ]:
X_val_v2_same = val_v3[model_features].copy()

for col in ["store_id", "item_id"]:
    X_val_v2_same[col] = X_val_v2_same[col].astype("category")

y_pred_v2_same = lgb_model_v2.predict(
    X_val_v2_same,
    num_iteration=lgb_model_v2.best_iteration_
)

v2_same_mae = mean_absolute_error(y_val_v3, y_pred_v2_same)
v2_same_rmse = np.sqrt(mean_squared_error(y_val_v3, y_pred_v2_same))

print(f"v2 same-data MAE:  {v2_same_mae:.4f}")
print(f"v2 same-data RMSE: {v2_same_rmse:.4f}")
print(f"v3 + lag52 MAE:    {v3_mae:.4f}")
print(f"v3 + lag52 RMSE:   {v3_rmse:.4f}")

In [ ]:
final_train_df = model_df_v3[
    model_df_v3["week_index"] <= 273
].copy()

final_test_df = model_df_v3[
    model_df_v3["week_index"].between(274, 277)
].copy()

print(
    "Final train:",
    final_train_df.shape,
    final_train_df["week_index"].min(),
    "-",
    final_train_df["week_index"].max()
)

print(
    "Final test:",
    final_test_df.shape,
    final_test_df["week_index"].min(),
    "-",
    final_test_df["week_index"].max()
)

In [ ]:
X_final_train = final_train_df[model_features_v3].copy()
y_final_train = final_train_df["weekly_sales"]

X_final_test = final_test_df[model_features_v3].copy()
y_final_test = final_test_df["weekly_sales"]

for col in ["store_id", "item_id"]:
    X_final_train[col] = X_final_train[col].astype("category")
    X_final_test[col] = X_final_test[col].astype("category")

In [ ]:
final_lgb_model = LGBMRegressor(
    objective="regression",
    n_estimators=977,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1
)

final_lgb_model.fit(
    X_final_train,
    y_final_train,
    categorical_feature=["store_id", "item_id"]
)

In [ ]:
y_test_pred = final_lgb_model.predict(X_final_test)

final_mae = mean_absolute_error(y_final_test, y_test_pred)
final_rmse = np.sqrt(
    mean_squared_error(y_final_test, y_test_pred)
)

# Aynı test setinde naive baseline
naive_test_pred = final_test_df["lag_1"]

naive_test_mae = mean_absolute_error(
    y_final_test,
    naive_test_pred
)

naive_test_rmse = np.sqrt(
    mean_squared_error(y_final_test, naive_test_pred)
)

print(f"Naive Test MAE:     {naive_test_mae:.4f}")
print(f"Naive Test RMSE:    {naive_test_rmse:.4f}")
print()
print(f"LightGBM Test MAE:  {final_mae:.4f}")
print(f"LightGBM Test RMSE: {final_rmse:.4f}")

In [ ]:
get_ipython().system('pip install optuna')

In [ ]:
import optuna
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
tune_train = train_v3[
    train_v3["week_index"] >= 170
].copy()

X_tune = tune_train[model_features_v3].copy()
y_tune = tune_train["weekly_sales"]

X_tune_val = val_v3[model_features_v3].copy()
y_tune_val = val_v3["weekly_sales"]

for col in ["store_id", "item_id"]:
    X_tune[col] = X_tune[col].astype("category")
    X_tune_val[col] = X_tune_val[col].astype("category")

print(X_tune.shape, X_tune_val.shape)

In [ ]:
def objective(trial):

    params = {
        "objective": "regression_l1",
        "n_estimators": 1500,

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.015, 0.08, log=True
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves", 31, 127
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 6, 14
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples", 20, 150
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.7, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.7, 1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-3, 2.0, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 2.0, log=True
        ),

        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1
    }

    model = LGBMRegressor(**params)

    model.fit(
        X_tune,
        y_tune,
        eval_set=[(X_tune_val, y_tune_val)],
        eval_metric="mae",
        categorical_feature=["store_id", "item_id"],
        callbacks=[
            lgb.early_stopping(75, verbose=False)
        ]
    )

    pred = model.predict(
        X_tune_val,
        num_iteration=model.best_iteration_
    )

    return mean_absolute_error(y_tune_val, pred)

In [ ]:
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(
    objective,
    n_trials=15,
    show_progress_bar=True
)

In [ ]:
print("Best MAE:", study.best_value)
print("\nBest parameters:")

for key, value in study.best_params.items():
    print(f"{key}: {value}")

In [ ]:
best_params = study.best_params.copy()

tuned_lgb = LGBMRegressor(
    objective="regression_l1",
    n_estimators=1500,
    **best_params,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

tuned_lgb.fit(
    X_train_v3,
    y_train_v3,
    eval_set=[(X_val_v3, y_val_v3)],
    eval_metric="mae",
    categorical_feature=["store_id", "item_id"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=75),
        lgb.log_evaluation(period=100)
    ]
)

In [ ]:
y_pred_tuned = tuned_lgb.predict(
    X_val_v3,
    num_iteration=tuned_lgb.best_iteration_
)

tuned_mae = mean_absolute_error(
    y_val_v3,
    y_pred_tuned
)

tuned_rmse = np.sqrt(
    mean_squared_error(y_val_v3, y_pred_tuned)
)

print(f"Naive:       MAE 3.9089")
print(f"LightGBM v3: MAE 3.2443 | RMSE 6.8727")
print(
    f"Tuned LGBM:  MAE {tuned_mae:.4f} | "
    f"RMSE {tuned_rmse:.4f}"
)
print(
    f"Best iteration: {tuned_lgb.best_iteration_}"
)

In [ ]:
final_tuned_lgb = LGBMRegressor(
    objective="regression_l1",
    n_estimators=1492,
    **study.best_params,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

final_tuned_lgb.fit(
    X_final_train,
    y_final_train,
    categorical_feature=["store_id", "item_id"]
)

In [ ]:
y_test_pred_tuned = final_tuned_lgb.predict(X_final_test)

tuned_test_mae = mean_absolute_error(
    y_final_test,
    y_test_pred_tuned
)

tuned_test_rmse = np.sqrt(
    mean_squared_error(y_final_test, y_test_pred_tuned)
)

print(f"Naive Test:       MAE 4.7234 | RMSE 10.9338")
print(f"LightGBM v3 Test: MAE 4.0473 | RMSE 9.2676")
print(
    f"Tuned LGBM Test:  MAE {tuned_test_mae:.4f} | "
    f"RMSE {tuned_test_rmse:.4f}"
)

In [ ]:
get_ipython().system(' pip install xgboost')

In [ ]:
X_train_xgb = X_train_v3.copy()
X_val_xgb = X_val_v3.copy()

for col in ["store_id", "item_id"]:
    categories = X_train_xgb[col].cat.categories

    X_train_xgb[col] = (
        X_train_xgb[col]
        .cat.codes
        .astype("int32")
    )

    X_val_xgb[col] = (
        X_val_xgb[col]
        .cat.set_categories(categories)
        .cat.codes
        .astype("int32")
    )

In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    objective="reg:absoluteerror",
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=10,
    min_child_weight=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_xgb,
    y_train_v3,
    eval_set=[(X_val_xgb, y_val_v3)],
    verbose=100
)

In [ ]:
xgb_val_pred = xgb_model.predict(X_val_xgb)

xgb_val_mae = mean_absolute_error(
    y_val_v3,
    xgb_val_pred
)

xgb_val_rmse = np.sqrt(
    mean_squared_error(y_val_v3, xgb_val_pred)
)

print(
    f"XGBoost Validation: "
    f"MAE {xgb_val_mae:.4f} | "
    f"RMSE {xgb_val_rmse:.4f}"
)

In [ ]:
X_tune_xgb = tune_train[model_features_v3].copy()
y_tune_xgb = tune_train["weekly_sales"]

X_tune_val_xgb = val_v3[model_features_v3].copy()
y_tune_val_xgb = val_v3["weekly_sales"]

for col in ["store_id", "item_id"]:
    X_tune_xgb[col] = X_tune_xgb[col].astype("category")
    X_tune_val_xgb[col] = X_tune_val_xgb[col].astype("category")

    categories = X_tune_xgb[col].cat.categories

    X_tune_xgb[col] = X_tune_xgb[col].cat.codes.astype("int32")

    X_tune_val_xgb[col] = (
        X_tune_val_xgb[col]
        .cat.set_categories(categories)
        .cat.codes
        .astype("int32")
    )

In [ ]:
def objective_xgb(trial):

    params = {
        "objective": "reg:absoluteerror",
        "n_estimators": 1500,

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.015, 0.08, log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 6, 14
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight", 5, 100
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.7, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.7, 1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-3, 2.0, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 5.0, log=True
        ),

        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1
    }

    model = XGBRegressor(**params)

    model.fit(
        X_tune_xgb,
        y_tune_xgb,
        eval_set=[(X_tune_val_xgb, y_tune_val_xgb)],
        verbose=False
    )

    pred = model.predict(X_tune_val_xgb)

    return mean_absolute_error(
        y_tune_val_xgb,
        pred
    )

In [ ]:
def objective_xgb(trial):

    params = {
        "objective": "reg:absoluteerror",
        "n_estimators": 1500,

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.015, 0.08, log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 6, 14
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight", 5, 100
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.7, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.7, 1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-3, 2.0, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 5.0, log=True
        ),

        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1
    }

    model = XGBRegressor(**params)

    model.fit(
        X_tune_xgb,
        y_tune_xgb,
        eval_set=[(X_tune_val_xgb, y_tune_val_xgb)],
        verbose=False
    )

    pred = model.predict(X_tune_val_xgb)

    return mean_absolute_error(
        y_tune_val_xgb,
        pred
    )

In [ ]:
study_xgb = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_xgb.optimize(
    objective_xgb,
    n_trials=15,
    show_progress_bar=True
)

print("Best XGBoost MAE:", study_xgb.best_value)

print("\nBest parameters:")
for key, value in study_xgb.best_params.items():
    print(f"{key}: {value}")

In [ ]:
from catboost import CatBoostRegressor

In [ ]:
get_ipython().system('pip install CatBoost')

In [ ]:
from catboost import CatBoostRegressor

In [ ]:
X_train_cat = train_v3[model_features_v3].copy()
y_train_cat = train_v3["weekly_sales"]

X_val_cat = val_v3[model_features_v3].copy()
y_val_cat = val_v3["weekly_sales"]

for col in ["store_id", "item_id"]:
    X_train_cat[col] = X_train_cat[col].astype(str)
    X_val_cat[col] = X_val_cat[col].astype(str)

In [ ]:
cat_model = CatBoostRegressor(
    loss_function="MAE",
    eval_metric="MAE",
    iterations=1000,
    learning_rate=0.05,
    depth=10,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=100,
    thread_count=-1,
    allow_writing_files=False
)

cat_model.fit(
    X_train_cat,
    y_train_cat,
    cat_features=["store_id", "item_id"],
    eval_set=(X_val_cat, y_val_cat),
    early_stopping_rounds=75,
    verbose=100
)